# 03 — Memory surfacing with a fake LTM

STM's headline feature is **proactive memory surfacing**:
when an upstream tool returns, STM queries a long-term
memory (LTM) server in the background and injects the most
relevant chunks at the top of the response. The agent sees
a single enriched response; no extra tool call needed.

This notebook shows surfacing end-to-end **without requiring
a real `memtomem-server`** — we point STM at
`notebooks/_fixtures/fake_ltm.py`, a notebook-local stub
that returns canned memories with per-call UUIDs so repeated
runs stay deterministic and aren't silently suppressed by
STM's cross-session dedup.

**You will learn:**

- How STM's surfacing engine talks to an LTM via MCP
- What an injected `<surfaced-memories>` block looks like
- How to send feedback back to STM with
  `stm_surfacing_feedback`
- Where to see surfacing counters in the stats

**Prereqs:** Notebook 01 completed.

## 1. Isolate state and point surfacing at the fake LTM

In [ ]:
# Add notebooks/ to sys.path so `_helpers` is importable regardless of
# where Jupyter was launched (from the repo root or from notebooks/).
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd / "notebooks" / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd / "notebooks"))
else:
    raise RuntimeError(
        "Cannot find notebooks/_helpers.py — run Jupyter from the repo "
        "root or from the notebooks/ directory."
    )

In [ ]:
import re
import subprocess

from _helpers import (
    isolate_stm_state,
    configure_fake_ltm,
    stm_session,
    extract_text,
    fixtures_dir,
)

# enable_surfacing=True leaves the surfacing env knobs alone
# so that configure_fake_ltm() can point them at the fake server.
config_path = isolate_stm_state(prefix="nb03_", enable_surfacing=True)
fake_ltm = configure_fake_ltm()
print(f"STM config → {config_path}")
print(f"Fake LTM   → {fake_ltm}")

> **Note:** In a real setup you would replace the fake LTM
> with a running `memtomem-server` process pointed at your
> actual memory store. The MCP protocol keeps the interface
> identical, so switching between fake and real is a single
> env var change.

## 2. Register a small upstream and talk to it via STM

In [ ]:
doc_script = fixtures_dir() / "doc_mcp.py"
subprocess.run(
    [
        "uv", "run", "mms", "add", "docfix",
        "--config", str(config_path),
        "--command", "uv",
        "--args", f"run python {doc_script}",
        "--prefix", "docfix",
        "--compression", "none",  # disable compression to highlight surfacing
    ],
    check=True, capture_output=True,
)
print("Registered docfix (compression disabled — we want raw + surfacing)")

In [ ]:
async with stm_session() as session:
    result = await session.call_tool("docfix__describe", {})
    enriched = extract_text(result)

print(enriched)

## 3. What just happened

The `docfix__describe` tool's own output is just:

```
docfix fixture — 8 sections: Overview, Installation, ...
```

But what came back above starts with a
`<surfaced-memories>` block containing two chunks from the
fake LTM — scored, labeled, and prepended to the real
response. STM did this without the agent asking. Notice
the **Surfacing ID** at the bottom of the block — that's
how the agent rates the surfacing.

## 4. Send feedback

In [ ]:
# Extract the surfacing ID from the response
match = re.search(r"Surfacing ID: ([a-f0-9]+)", enriched)
if not match:
    raise RuntimeError("Surfacing ID not found — surfacing did not fire")
surfacing_id = match.group(1)
print(f"Surfacing ID: {surfacing_id}")

async with stm_session() as session:
    feedback_result = await session.call_tool(
        "stm_surfacing_feedback",
        {"surfacing_id": surfacing_id, "rating": "helpful"},
    )
    print(extract_text(feedback_result))

Over time, `"helpful"` feedback raises the confidence of the
retrieved memory chunks via STM's auto-tuning loop, and
`"not_relevant"` pulls them down. Agents that call the
feedback tool get progressively more relevant surfacing.
See [`docs/surfacing.md`](../docs/surfacing.md) for the
scoring details.

> **S1 failure guard.** In production, if `record_surfacing`
> fails (e.g. SQLite contention on `stm_feedback.db`), STM
> drops the `surfacing_id` — the memory block is still
> injected but without a feedback ID. When this happens,
> `stm_surfacing_feedback` cannot be called for that event.
> The response is never blocked. See
> [`docs/pipeline.md → Stage 3`](../docs/pipeline.md#stage-3-surface).

## 5. Check the surfacing counters

In [ ]:
async with stm_session() as session:
    stats_result = await session.call_tool("stm_surfacing_stats", {})
    print(extract_text(stats_result))

## Where to next

- **Horizontal scaling (scenario 5 from the docs)** — STM
  can share a SQLite-backed pending store across multiple
  instances so that an agent can start a selective
  compression on instance A and resolve it on instance B.
  Not demonstrable in a single-process notebook; see
  [`docs/operations.md#horizontal-scaling`](../docs/operations.md#horizontal-scaling).
- **Notebook 04** — put a real LangChain / LangGraph agent
  in front of STM using `create_agent` and
  `langchain-mcp-adapters`. That notebook needs an Anthropic
  or OpenAI API key, so it's skipped in CI by default.